In [3]:
from pathlib import Path
import re
import pandas as pd

# ============================================================
# Configuration
# ============================================================

HIGH_RISK_FILE = Path(
    r"D:/12. RQ2 eDySec/Adversarial Attacks/high_risk_packages_107_csv/selected_packages.csv"
)

TRACE_FILE = Path(
    r"D:/12. RQ2 eDySec/Adversarial Attacks/FinalCombinedBenignMaliciousDataset-v4 36F.csv"
)

OUTPUT_FILE = Path(
    r"D:/12. RQ2 eDySec/Adversarial Attacks/107_Packages_Traces.csv"
)

OUTPUT_CSV = OUTPUT_FILE.with_suffix(".csv")

ARCHIVE_SUFFIXES = (
    ".tar.gz", ".tar.bz2", ".tar.xz",
    ".tgz", ".tbz2", ".txz",
    ".zip", ".whl", ".egg", ".tar",
    ".gz", ".bz2", ".xz",
)

# ============================================================
# Normalization
# ============================================================

def strip_archive_suffix(value: str) -> str:
    value = str(value).strip()
    lowered = value.lower()

    for suffix in ARCHIVE_SUFFIXES:
        if lowered.endswith(suffix):
            return value[:-len(suffix)]

    return value


def canonicalize(value: str) -> str:
    return re.sub(
        r"[-_.\s]+",
        "-",
        str(value).strip().lower(),
    ).strip("-")


# ============================================================
# Extraction
# ============================================================

def extract_selected_traces(
    high_risk_file: str | Path = HIGH_RISK_FILE,
    trace_file: str | Path = TRACE_FILE,
    output_file: str | Path = OUTPUT_FILE,
):
    high_risk_file = Path(high_risk_file)
    trace_file = Path(trace_file)
    output_file = Path(output_file)

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    selected = pd.read_csv(
        high_risk_file,
        dtype=str,
        keep_default_na=False,
    )

    traces = pd.read_csv(
        trace_file,
        dtype=str,
        keep_default_na=False,
        low_memory=False,
    )

    required_selected = {
        "Package Name",
        "Package Version",
    }

    if not required_selected.issubset(selected.columns):
        raise KeyError(
            "high_risk_details.csv must contain "
            "'Package Name' and 'Package Version'."
        )

    if "Package_Name" not in traces.columns:
        raise KeyError(
            "Trace CSV must contain 'Package_Name'."
        )

    original_trace_columns = traces.columns.tolist()

    traces["__stem"] = traces["Package_Name"].map(
        strip_archive_suffix
    )
    traces["__stem_lower"] = (
        traces["__stem"].str.lower()
    )
    traces["__canonical_stem"] = (
        traces["__stem"].map(canonicalize)
    )

    matched_indexes = []
    audit_rows = []
    unmatched = []
    ambiguous = []

    for selection_number, row in selected.iterrows():
        package_name = row["Package Name"].strip()
        package_version = row["Package Version"].strip()

        expected = f"{package_name}-{package_version}"
        expected_lower = expected.lower()
        expected_canonical = canonicalize(expected)

        candidates = traces.index[
            traces["__stem_lower"].eq(expected_lower)
        ].tolist()
        method = "Exact name-version"

        if not candidates:
            candidates = traces.index[
                traces["__stem_lower"].str.startswith(
                    expected_lower + "-"
                )
            ].tolist()
            method = "Name-version prefix"

        if not candidates:
            candidates = traces.index[
                traces["__canonical_stem"].eq(
                    expected_canonical
                )
            ].tolist()
            method = "Canonicalized name-version"

        if len(candidates) == 0:
            unmatched.append(expected)
            continue

        if len(candidates) > 1:
            ambiguous.append(
                {
                    "Expected": expected,
                    "Candidates": traces.loc[
                        candidates,
                        "Package_Name",
                    ].tolist(),
                }
            )
            continue

        trace_index = candidates[0]
        matched_indexes.append(trace_index)

        audit_rows.append(
            {
                "Selection Number": selection_number + 1,
                "Package Name": package_name,
                "Package Version": package_version,
                "High-Risk Category": row.get(
                    "High-Risk Category",
                    "",
                ),
                "Matched Trace Package_Name": traces.at[
                    trace_index,
                    "Package_Name",
                ],
                "Match Method": method,
                "Status": "Matched",
            }
        )

    if unmatched or ambiguous:
        raise RuntimeError(
            "Trace matching was not one-to-one.\n"
            f"Unmatched: {unmatched}\n"
            f"Ambiguous: {ambiguous}"
        )

    matched_traces = traces.loc[
        matched_indexes,
        original_trace_columns,
    ].reset_index(drop=True)

    audit = pd.DataFrame(audit_rows)

    if len(matched_traces) != len(selected):
        raise RuntimeError(
            f"Expected {len(selected)} traces, "
            f"but matched {len(matched_traces)}."
        )

    summary = pd.DataFrame(
        [
            {
                "Metric": "Selected high-risk records",
                "Value": len(selected),
            },
            {
                "Metric": "Matched trace rows",
                "Value": len(matched_traces),
            },
            {
                "Metric": "Trace columns retained",
                "Value": len(original_trace_columns),
            },
            {
                "Metric": "Unmatched records",
                "Value": len(unmatched),
            },
            {
                "Metric": "Ambiguous records",
                "Value": len(ambiguous),
            },
        ]
    )

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl",
    ) as writer:
        matched_traces.to_excel(
            writer,
            sheet_name="Selected 64 Traces",
            index=False,
        )
        audit.to_excel(
            writer,
            sheet_name="Match Audit",
            index=False,
        )
        selected.to_excel(
            writer,
            sheet_name="High Risk Details",
            index=False,
        )
        summary.to_excel(
            writer,
            sheet_name="Summary",
            index=False,
        )

    matched_traces.to_csv(
        output_file.with_suffix(".csv"),
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Selected records : {len(selected)}")
    print(f"Matched traces   : {len(matched_traces)}")
    print(f"Trace columns    : {len(original_trace_columns)}")
    print(f"Output Excel     : {output_file}")
    print(f"Output CSV       : {output_file.with_suffix('.csv')}")

    return {
        "traces": matched_traces,
        "audit": audit,
        "selected": selected,
        "summary": summary,
    }


if __name__ == "__main__":
    extract_selected_traces()


Selected records : 107
Matched traces   : 107
Trace columns    : 37
Output Excel     : D:\12. RQ2 eDySec\Adversarial Attacks\107_Packages_Traces.csv
Output CSV       : D:\12. RQ2 eDySec\Adversarial Attacks\107_Packages_Traces.csv
